In [1]:
#import library
from pandas_gbq import read_gbq
import pandas as pd
import numpy as np
import os
import datetime
import ssl
import logging

In [2]:
from datetime import date, timedelta
from dateutil.relativedelta import relativedelta

# Get today's date
today = date.today()
idx = (today.weekday() + 1) % 7

# Current week (CY) range
end_date = today - timedelta(days=(7 + idx - 6))
start_date = end_date - timedelta(days=6)

# Prior period (PP) range
pp_end_date = start_date - timedelta(days=1)
pp_start_date = pp_end_date - timedelta(days=13)

# Corresponding week last year (CWLY and PWLY)
cwly_date = pp_end_date - timedelta(days=364) + timedelta(days=7)
pwly_date = pp_end_date - timedelta(days=364)

ytd_last_year = end_date - relativedelta(years=1)
begin_of_current_year = date(end_date.year, 1, 1)
begin_of_last_year = date(end_date.year - 1, 1, 1)

# Output all dates
(end_date, start_date, pp_end_date, pp_start_date, cwly_date, pwly_date,begin_of_last_year,ytd_last_year,begin_of_current_year)


(datetime.date(2026, 1, 31),
 datetime.date(2026, 1, 25),
 datetime.date(2026, 1, 24),
 datetime.date(2026, 1, 11),
 datetime.date(2025, 2, 1),
 datetime.date(2025, 1, 25),
 datetime.date(2025, 1, 1),
 datetime.date(2025, 1, 31),
 datetime.date(2026, 1, 1))

In [3]:
from custom_query import read_sql_query, sql_files

# Read SQL queries from files
queries = {key: read_sql_query(path) for key, path in sql_files.items()}

# Execute queries if all were successfully reada
if all(queries.values()):
    try:
        weekly_note = read_gbq(queries["weekly_note"], project_id='pcln-pl-airanalytics-prod')
        finance_data = read_gbq(queries["finance_data"], project_id='pcln-pl-airanalytics-prod')
        gds_incentives = read_gbq(queries["gds_incentives"], project_id='pcln-pl-airanalytics-prod')
        tsa_data = read_gbq(queries["tsa_data"], project_id='pcln-pl-airanalytics-prod')
        deal_share = read_gbq(queries["deal_share"], project_id='pcln-pl-airanalytics-prod')
        upsell_data = read_gbq(queries["upsell_query"], project_id='pcln-pl-airanalytics-prod')
        direct_parity_data = read_gbq(queries["direct_parity_data"], project_id='pcln-pl-airanalytics-prod')
        meta_parity_data = read_gbq(queries["meta_parity_data_query"], project_id='pcln-pl-airanalytics-prod')
        bookability_data = read_gbq(queries["bookability_query"], project_id='pcln-pl-airanalytics-prod')
        roi_data = read_gbq(queries["roi_query"], project_id='pcln-pl-airanalytics-prod')
        dau_conversion_data = read_gbq(queries["dau_conversion_query"], project_id='pcln-pl-airanalytics-prod')
        # midt_data = read_gbq(queries["midt_query"], project_id='pcln-pl-airanalytics-prod')
        sem_data = read_gbq(queries["sem_query"], project_id='pcln-pl-airanalytics-prod')
        print("SQL queries executed successfully.")
        
    except Exception as e:
        print(f"Failed to execute SQL queries: {e}")

/Users/sye/Library/Python/3.9/lib/python/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: |          |
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
Downloading: 100%|██████████|
SQL queries executed successfully.


## make a copy of dataset

In [4]:
# make a copy of every data set
df_weekly=weekly_note.copy()
df_finance=finance_data.copy()
df_gds_incentive=gds_incentives.copy()
df_tsa=tsa_data.copy()
df_direct_parity=direct_parity_data.copy()
df_meta_parity = meta_parity_data.copy()
df_roi=roi_data.copy()
dau_conversion=dau_conversion_data.copy()
df_deal_share=deal_share.copy()
df_upsell=upsell_data.copy()
df_bookability=bookability_data.copy()
df_weekly.columns = df_weekly.columns.str.lower()
# df_midt=midt_data.copy()
df_sem=sem_data.copy()


In [5]:

base_dir = '../../data_file/'

# Check if directory exists
if not os.path.exists(base_dir):
    print(f"Error: Directory {base_dir} does not exist!")

# Save files
file_names = {
    'df_weekly': df_weekly,
    'df_finance': df_finance,
    'df_gds_incentive':df_gds_incentive,
    'df_tsa': df_tsa,
    'direct_parity': df_direct_parity,
    # 'meta_parity': df_meta_parity,
    # 'dau_conversion': dau_conversion,
    # 'df_roi': df_roi,
    'deal_share': df_deal_share,
    'df_upsell':df_upsell,
    'bookability': df_bookability,
 
}

# Save with verification
for name, df in file_names.items():
    file_path = f'{base_dir}{name}.csv'
    print(f"Saving: {file_path}")
    # print(df.head())  # Print first few rows to confirm data
    df.to_csv(file_path, index=False)

# Read files
for name in file_names.keys():
    file_path = f'{base_dir}{name}.csv'
    print(f"Reading: {file_path}")

    if os.path.exists(file_path):
        df_read = pd.read_csv(file_path)
        # print(df_read.head())  # Print first few rows to compare
    else:
        print(f"Error: {file_path} not found!")

Saving: ../../data_file/df_weekly.csv
Saving: ../../data_file/df_finance.csv
Saving: ../../data_file/df_gds_incentive.csv
Saving: ../../data_file/df_tsa.csv
Saving: ../../data_file/direct_parity.csv
Saving: ../../data_file/deal_share.csv
Saving: ../../data_file/df_upsell.csv
Saving: ../../data_file/bookability.csv
Reading: ../../data_file/df_weekly.csv
Reading: ../../data_file/df_finance.csv
Reading: ../../data_file/df_gds_incentive.csv
Reading: ../../data_file/df_tsa.csv
Reading: ../../data_file/direct_parity.csv
Reading: ../../data_file/deal_share.csv
Reading: ../../data_file/df_upsell.csv
Reading: ../../data_file/bookability.csv


In [6]:
# import os
# import pandas as pd

# base_dir = '../../data_file/'

# # Just a list of names that match the filenames you saved
# file_names = [
#     'df_weekly',
#     'df_finance',
#     'df_tsa',
#     'direct_parity',
#     'meta_parity',
#     'dau_conversion',
#     'df_roi',
#     'deal_share',
#     # 'bookability',
# ]


# for name in file_names:
#     file_path = f'{base_dir}{name}.csv'
#     print(f"Reading: {file_path}")
    
#     if os.path.exists(file_path):
#         globals()[name] = pd.read_csv(file_path)  # creates df_weekly, df_finance, ...
#     else:
#         print(f"Error: {file_path} not found!")



## Summary Table

In [7]:

def format_number(num):
    if pd.isna(num):
        return ''
    if abs(num) >= 1e6:
        return f"{num/1e6:.1f}M"
    elif abs(num) >= 1e3:
        return f"{num/1e3:.0f}K"
    elif abs(num) < 1e3:
        return "<1K"
    return f"{num:.0f}"


def format_percentage(x, decimals=1, multiply_100=True):
    def fmt(v):
        if pd.isna(v):
            return ''
        v = float(v)
        if multiply_100:
            v *= 1
        return f"{v:.{decimals}f}%"

    if isinstance(x, pd.Series):
        return x.map(fmt)
    if isinstance(x, pd.DataFrame):
        return x.applymap(fmt)  # or: x.map(fmt) in newer pandas
    return fmt(x)  # scalar (int/float/np.float64)


def format_percentage_2(num):
    if pd.isna(num):
        return ''
    return f"{num:.2f}%"

def round_to_nearest_10(num):
  return round(num / 10) * 10

df_pricelince=df_weekly[(df_weekly['brand']== 'Priceline')& (df_weekly['wk_ending']>= pp_end_date)& (df_weekly['wk_ending']<= end_date)]
df_pricelince_air=df_weekly[(df_weekly['brand']== 'Priceline')&(df_weekly['offer_type']== 'Flights Only')& (df_weekly['wk_ending']>= pp_end_date)& (df_weekly['wk_ending']<= end_date)]
df_pricelince_b2c=df_weekly[(df_weekly['brand']== 'Priceline')&(df_weekly['company']== 'Priceline B2C')&(df_weekly['wk_ending']>= pp_end_date)& (df_weekly['wk_ending']<= end_date)]
df_pricelince_b2c_standalone=df_weekly[(df_weekly['brand']== 'Priceline')&(df_weekly['company']== 'Priceline B2C')&(df_weekly['offer_type']== 'Flights Only')&(df_weekly['wk_ending']>= pp_end_date)& (df_weekly['wk_ending']<= end_date)]


In [8]:
import kpi
import importlib

importlib.reload(kpi)

from kpi import calculate_business_metrics

# Calculate business metrics
df_business = calculate_business_metrics(df_pricelince,end_date, pp_end_date)

from kpi import calculate_carrier_metrics
# Calculate carrier metrics
df_carrier = calculate_carrier_metrics(df_pricelince_b2c_standalone, end_date, pp_end_date)

from kpi import calculate_channel_metrics
# Calculate channel metrics
df_channel = calculate_channel_metrics(df_pricelince_b2c_standalone, end_date, pp_end_date)

from kpi import calculate_source_metrics
# Calculate source metrics
df_source = calculate_source_metrics(df_pricelince_b2c_standalone, end_date, pp_end_date)

from kpi import calculate_brand_metrics
# Calculate brand metrics
df_brand = calculate_brand_metrics(df_weekly, end_date, pp_end_date)


In [9]:
df_business

,Net Tickets_standalone,YoY_standalone,YoY PW_standalone,Net Tickets_package,YoY_package,YoY PW_package,Net Tickets_total,YoY_total,YoY PW_total
company,,,,,,,,,
B2C,140K,-1.3%,-1.1%,30K,13.8%,29.5%,171K,1.1%,3.6%
B2B,6K,-10.5%,-23.9%,6K,-2.2%,-2.6%,13K,-6.5%,-13.9%
Total,147K,-1.7%,-2.2%,37K,10.7%,23.1%,184K,0.6%,2.3%


In [10]:
df_channel


,Net Tickets_App,YoY_App,YoY PW_App,Net Tickets_Desk/MWEB,YoY_Desk/MWEB,YoY PW_Desk/MWEB,Net Tickets_Total,YoY_Total,YoY PW_Total
search_channel_group,,,,,,,,,
Direct,30K,1.9%,2.5%,19K,-24.8%,-25.9%,49K,-10.5%,-10.9%
Web Marketing,8K,115.2%,110.8%,52K,3.4%,4.4%,60K,11.4%,12.4%
Shop PPC,5K,-9.7%,-7.8%,17K,-7.2%,-7.4%,22K,-7.8%,-7.5%
Affiliate,<1K,202.2%,289.6%,9K,-5.5%,-6.2%,9K,-3.6%,-4.1%
Total,44K,11.8%,12.4%,97K,-6.2%,-6.3%,140K,-1.3%,-1.1%


In [11]:
df_carrier

,Net Tickets_Retail,YoY_Retail,YoY PW_Retail,Net Tickets_Opaque,YoY_Opaque,YoY PW_Opaque,Net Tickets_Total,YoY_Total,YoY PW_Total
carrier,,,,,,,,,
American Airlines (AA),23K,-22.2%,-16.2%,1K,-85.2%,-86.1%,24K,-34.2%,-29.7%
Delta Air Lines (DL),17K,-21.4%,-16.1%,<1K,-88.7%,-88.3%,18K,-25.0%,-21.0%
United Airlines (UA),15K,-0.4%,-1.7%,8K,15.5%,1.0%,23K,4.8%,-0.7%
Southwest Airlines (WN),18K,17297.0%,15005.4%,<1K,inf%,inf%,18K,17402.0%,15102.7%
Spirit Airlines (NK),10K,-38.9%,-48.9%,,,,10K,-38.9%,-48.9%
Frontier Airlines (F9),18K,38.5%,33.1%,,,,18K,38.5%,33.1%
Alaska Airlines (AS),5K,18.5%,30.2%,3K,27.2%,49.7%,8K,21.9%,38.2%
JetBlue Airways (B6),5K,-11.2%,5.1%,<1K,900.0%,inf%,5K,-11.0%,5.5%
Other,16K,-10.9%,-13.8%,<1K,423.8%,883.3%,17K,-10.4%,-13.0%


In [12]:
df_source

,Net Tickets_Published,YoY_Published,YoY PW_Published,Net Tickets_Private,YoY_Private,YoY PW_Private,Net Tickets_Total,YoY_Total,YoY PW_Total
gds_booking_category,,,,,,,,,
Direct Connect,61K,3.8%,1.9%,1K,-80.3%,-80.3%,63K,-4.8%,-6.8%
Indirect Connect,62K,0.2%,4.0%,15K,9.8%,5.0%,77K,2.0%,4.2%
Phone Sales,,,,1K,-7.0%,-16.7%,1K,-7.0%,-16.7%
Total,123K,2.0%,2.9%,17K,-19.6%,-21.7%,140K,-1.3%,-1.1%


# MOR


In [13]:
# Create df_mor
df_mor= pd.DataFrame()
df_mor['Net Tickets'] = df_pricelince[(df_pricelince['wk_ending'] == end_date)& ((df_pricelince['merchant_of_record'] == 'VCC')|(df_pricelince['merchant_of_record'] == 'PCLN'))].groupby(['offer_method_code'])['net_tkts_cy'].sum().round(-1).astype(int)
df_mor['Net Tickets_cwly'] = df_pricelince[(df_pricelince['wk_ending'] == end_date)& ((df_pricelince['merchant_of_record'] == 'VCC')|(df_pricelince['merchant_of_record'] == 'PCLN'))].groupby(['offer_method_code'])['net_tkts_ly'].sum().round(-1).astype(int)
df_mor['Net Tickets_PW'] = df_pricelince[(df_pricelince['wk_ending'] == pp_end_date)& ((df_pricelince['merchant_of_record'] == 'VCC')|(df_pricelince['merchant_of_record'] == 'PCLN'))].groupby(['offer_method_code'])['net_tkts_cy'].sum().round(-1).astype(int)
df_mor['Net Tickets_PWly'] = df_pricelince[(df_pricelince['wk_ending'] == pp_end_date)& ((df_pricelince['merchant_of_record'] == 'VCC')|(df_pricelince['merchant_of_record'] == 'PCLN'))].groupby(['offer_method_code'])['net_tkts_ly'].sum().round(-1).astype(int)


# Calculate YoY and YoY PW
df_mor.loc['Total','Net Tickets':'Net Tickets_PWly']=[df_mor['Net Tickets'].sum(),df_mor['Net Tickets_cwly'].sum(),df_mor['Net Tickets_PW'].sum(), df_mor['Net Tickets_PWly'].sum()]


# Create df_mor_1
df_mor_1= pd.DataFrame()
df_mor_1['Net Tickets'] = df_pricelince[(df_pricelince['wk_ending'] == end_date)].groupby(['offer_method_code'])['net_tkts_cy'].sum().round(-1).astype(int)
df_mor_1['Net Tickets_cwly'] = df_pricelince[(df_pricelince['wk_ending'] == end_date)].groupby(['offer_method_code'])['net_tkts_ly'].sum().round(-1).astype(int)
df_mor_1['Net Tickets_PW'] = df_pricelince[(df_pricelince['wk_ending'] == pp_end_date)].groupby(['offer_method_code'])['net_tkts_cy'].sum().round(-1).astype(int)
df_mor_1['Net Tickets_PWly'] = df_pricelince[(df_pricelince['wk_ending'] == pp_end_date)].groupby(['offer_method_code'])['net_tkts_ly'].sum().round(-1).astype(int)
# Calculate YoY and YoY PW
df_mor_1.loc['Total','Net Tickets':'Net Tickets_PWly']=[df_mor_1['Net Tickets'].sum(),df_mor_1['Net Tickets_cwly'].sum(),df_mor_1['Net Tickets_PW'].sum(), df_mor_1['Net Tickets_PWly'].sum()]

df_mor = pd.concat([df_mor,df_mor_1.add_suffix('_total')], axis=1)


actual_cy=(df_mor['Net Tickets']/df_mor['Net Tickets_total'])
actual_ly=(df_mor['Net Tickets_cwly']/df_mor['Net Tickets_cwly_total'])
actual_pw=(df_mor['Net Tickets_PW']/df_mor['Net Tickets_PW_total'])
actual_pwly=(df_mor['Net Tickets_PWly']/df_mor['Net Tickets_PWly_total'])
df_mor['Actual']=format_percentage(actual_cy*100)
df_mor['YoY']=format_percentage(((actual_cy/actual_ly)-1)*100)
df_mor['YoY_PW']=format_percentage(((actual_pw/actual_pwly)-1)*100)




##  US Outbound

In [14]:
# Create us Outbound
df_us_outbound= pd.DataFrame()
df_us_outbound['Net Tickets'] = df_pricelince[(df_pricelince['wk_ending'] == end_date)].groupby(['us_travel_type'])['net_tkts_cy'].sum().astype(int)
df_us_outbound['Net Tickets_cwly'] = df_pricelince[(df_pricelince['wk_ending'] == end_date)].groupby(['us_travel_type'])['net_tkts_ly'].sum().astype(int)
df_us_outbound['Net Tickets_pw'] = df_pricelince[(df_pricelince['wk_ending'] == pp_end_date)].groupby(['us_travel_type'])['net_tkts_cy'].sum().astype(int)
df_us_outbound['Net Tickets_pwly'] = df_pricelince[(df_pricelince['wk_ending'] == pp_end_date)].groupby(['us_travel_type'])['net_tkts_ly'].sum().astype(int)

# df_us_outbound['Net Tickets_pwly'].sum()
df_us_outbound.loc['Total','Net Tickets':'Net Tickets_pwly']=[df_us_outbound['Net Tickets'].sum(),df_us_outbound['Net Tickets_cwly'].sum(),df_us_outbound['Net Tickets_pw'].sum(), df_us_outbound['Net Tickets_pwly'].sum()]

actual_cy=(df_us_outbound.loc['US Outbound','Net Tickets']*100/df_us_outbound.loc['Total','Net Tickets'])
actual_ly=df_us_outbound.loc['US Outbound','Net Tickets_cwly']*100/df_us_outbound.loc['Total','Net Tickets_cwly']
actual_pw=df_us_outbound.loc['US Outbound','Net Tickets_pw']*100/df_us_outbound.loc['Total','Net Tickets_pw']
actual_pwly=df_us_outbound.loc['US Outbound','Net Tickets_pwly']*100/df_us_outbound.loc['Total','Net Tickets_pwly']

actual_cy,actual_ly,actual_pw,actual_pwly

(np.float64(9.76146629035949),
 np.float64(10.12749776728233),
 np.float64(9.992002867937016),
 np.float64(10.560439684456432))

## Share of buiness

In [15]:
df_share_business= pd.DataFrame()
df_share_business=df_mor.loc[['Retail (Disclosed)', 'Total'],['Actual','YoY','YoY_PW']]
df_share_business

df_share_business.loc['US Outbound','Actual']=format_percentage(actual_cy)

df_share_business.loc['US Outbound','YoY']=format_percentage(((actual_cy/actual_ly)-1)*100)

df_share_business.loc['US Outbound','YoY_PW']=format_percentage(((actual_pw/actual_pwly)-1)*100)
new_index_order = ['US Outbound','Retail (Disclosed)','Total']
df_share_business=df_share_business.reindex(new_index_order).fillna('')
df_share_business

,Actual,YoY,YoY_PW
offer_method_code,,,
US Outbound,9.8%,-3.6%,-5.4%
Retail (Disclosed),15.0%,81.8%,67.9%
Total,21.0%,21.8%,14.4%


##  DAU / ROI


In [16]:

from dau_roi_table import calculate_dau_conversion
from dau_roi_table import calculate_roi
from dau_roi_table import create_roi_table

import importlib, dau_roi_table

importlib.reload(dau_roi_table)


# 1) Compute raw DAU/Conversion
df_dau_converison= calculate_dau_conversion(dau_conversion,format_percentage,end_date, pp_end_date, cwly_date, pwly_date,begin_of_current_year,begin_of_last_year,ytd_last_year)
df_dau_converison


,DAU,DAU_cwly,DAU_pw,DAU_pwly,DAU_ytd,DAU_ytd_ly,converted,converted_cwly,converted_pw,converted_pwly,...,converted YoY_PW,DAU YoY,DAU YoY_PW,DAU YoY_YTD,conversion,conversion_cwly,conversion_pw,conversion_pwly,conversion_YoY,conversion_YoY_PW
channel,,,,,,,,,,,,,,,,,,,,,
Affiliate,12272,6635,12997,6599,53726,29825,1787,1050,1772,1037,...,70.877531,85.0%,97.0%,80.1%,0.145616,0.158252,0.136339,0.157145,-8.0%,-13.2%
Direct,799927,860715,811349,870971,3554766,4023438,40561,44383,40495,43580,...,-7.078935,-7.1%,-6.8%,-11.6%,0.050706,0.051565,0.049911,0.050036,-1.7%,-0.3%
SEM Brand,144303,151930,146297,153471,596012,704199,8738,9533,8735,9307,...,-6.145912,-5.0%,-4.7%,-15.4%,0.060553,0.062746,0.059707,0.060643,-3.5%,-1.5%
SEM Core,511367,418694,526987,423353,2103797,1967059,17529,15359,16877,15031,...,12.281285,22.1%,24.5%,7.0%,0.034279,0.036683,0.032025,0.035505,-6.6%,-9.8%
Shop PPC Cheapflights,224033,250847,225546,233988,908840,1137616,2709,3635,2626,3313,...,-20.736493,-10.7%,-3.6%,-20.1%,0.012092,0.014491,0.011643,0.014159,-16.6%,-17.8%
Shop PPC Google,573,447,555,507,2219,2244,25,24,20,12,...,66.666667,28.2%,9.5%,-1.1%,0.04363,0.053691,0.036036,0.023669,-18.7%,52.3%
Shop PPC Kayak,166925,43961,166314,46089,634509,214811,2627,1203,2393,1321,...,81.150643,279.7%,260.9%,195.4%,0.015738,0.027365,0.014388,0.028662,-42.5%,-49.8%
Shop PPC Others,132373,132988,128785,129975,539329,577391,4943,4962,4858,4651,...,4.450656,-0.5%,-0.9%,-6.6%,0.037341,0.037312,0.037722,0.035784,0.1%,5.4%
Total,1991773,1866217,2018830,1864953,8393198,8656583,78919,80149,77776,78252,...,-0.608291,6.7%,8.3%,-3.0%,0.039622,0.042947,0.038525,0.041959,-7.7%,-8.2%


In [17]:
import pandas as pd

df_roi = df_roi.copy()
df_roi["week_ending"] = pd.to_datetime(df_roi["week_ending"], errors="coerce")

end_date    = pd.to_datetime(end_date)
pp_end_date = pd.to_datetime(pp_end_date)
cwly_date   = pd.to_datetime(cwly_date)
pwly_date   = pd.to_datetime(pwly_date)

def sum_by_channel_roi(dt, col):
    m = df_roi["week_ending"].eq(dt)
    return df_roi.loc[m].groupby("channel")[col].sum()

df_roi_section = pd.DataFrame()

# contribution
df_roi_section["contribution"]      = sum_by_channel_roi(end_date,     "contribution")
df_roi_section["contribution_cwly"] = sum_by_channel_roi(cwly_date,    "contribution")
df_roi_section["contribution_pw"]   = sum_by_channel_roi(pp_end_date,  "contribution")
df_roi_section["contribution_pwly"] = sum_by_channel_roi(pwly_date,    "contribution")

# cost
df_roi_section["cost"]      = sum_by_channel_roi(end_date,     "cost")
df_roi_section["cost_cwly"] = sum_by_channel_roi(cwly_date,    "cost")
df_roi_section["cost_pw"]   = sum_by_channel_roi(pp_end_date,  "cost")
df_roi_section["cost_pwly"] = sum_by_channel_roi(pwly_date,    "cost")

# ROI
df_roi_section["ROI"]      = df_roi_section["contribution"]      / df_roi_section["cost"].replace(0, pd.NA)
df_roi_section["ROI_cwly"] = df_roi_section["contribution_cwly"] / df_roi_section["cost_cwly"].replace(0, pd.NA)
df_roi_section["ROI_pw"]   = df_roi_section["contribution_pw"]   / df_roi_section["cost_pw"].replace(0, pd.NA)
df_roi_section["ROI_pwly"] = df_roi_section["contribution_pwly"] / df_roi_section["cost_pwly"].replace(0, pd.NA)

# ROI YoY (% numeric)
df_roi_section["ROI_YoY"]    = (df_roi_section["ROI"]    / df_roi_section["ROI_cwly"].replace(0, pd.NA) - 1) * 100
df_roi_section["ROI_YoY_PW"] = (df_roi_section["ROI_pw"] / df_roi_section["ROI_pwly"].replace(0, pd.NA) - 1) * 100

# totals row
df_roi_section.loc["Total", df_roi_section.columns] = df_roi_section.sum(numeric_only=True)

df_roi_section


# rename ROI channels to match DAU conversion channels
df_roi_section2 = df_roi_section.rename(index={
    "CHEAPFLIGHTS": "Shop PPC Cheapflights",
    "CJ": "Affiliate",
    "CLICKTRIPZ": "Shop PPC Others",
    "SKYSCANNER": "Shop PPC Skyscanner",
    "Kayak": "Shop PPC Kayak",
})

dau = df_dau_converison.reset_index()                 # has: channel + DAU/conv cols
roi = df_roi_section2.reset_index().rename(columns={"index": "channel"})  # has: channel + ROI cols

merged = dau.merge(roi, on="channel", how="left").set_index("channel")

df_roi_v = merged[[
    "DAU", "DAU YoY", "DAU YoY_PW",
    "conversion", "conversion_YoY", "conversion_YoY_PW",
    "ROI", "ROI_YoY", "ROI_YoY_PW",
]].copy()

# IMPORTANT: avoid duplicate col names
df_roi_v.columns = [
    "DAU", "DAU_YoY", "DAU_YoY_PW",
    "Conversion", "Conversion_YoY", "Conversion_YoY_PW",
    "ROI", "ROI_YoY", "ROI_YoY_PW",
]

# format DAU / ROI (keep conversion numeric for now)
df_roi_v["DAU"] = df_roi_v["DAU"].apply(format_number)
df_roi_v["ROI"] = df_roi_v["ROI"].apply(lambda x: "" if pd.isna(x) else f"{x:.2f}")

order = [
    "Direct", "SEM Core", "SEM Brand",
    "Shop PPC Cheapflights", "Shop PPC Google", "Shop PPC Kayak",
    "Shop PPC Skyscanner", "Shop PPC Others",
    "Affiliate", "Total",
]
df_roi_v = df_roi_v.reindex(order)

df_roi_v


,DAU,DAU_YoY,DAU_YoY_PW,Conversion,Conversion_YoY,Conversion_YoY_PW,ROI,ROI_YoY,ROI_YoY_PW
channel,,,,,,,,,
Direct,800K,-7.1%,-6.8%,0.050706,-1.7%,-0.3%,,NaN,NaN
SEM Core,511K,22.1%,24.5%,0.034279,-6.6%,-9.8%,1.01,0.667222,-2.082981
SEM Brand,144K,-5.0%,-4.7%,0.060553,-3.5%,-1.5%,7.00,1.316138,-3.094303
Shop PPC Cheapflights,224K,-10.7%,-3.6%,0.012092,-16.6%,-17.8%,1.06,3.054657,-7.620976
Shop PPC Google,<1K,28.2%,9.5%,0.04363,-18.7%,52.3%,,NaN,NaN
Shop PPC Kayak,167K,279.7%,260.9%,0.015738,-42.5%,-49.8%,1.68,18.383483,38.457463
Shop PPC Skyscanner,NaN,NaN,NaN,<NA>,NaN,NaN,NaN,NaN,NaN
Shop PPC Others,132K,-0.5%,-0.9%,0.037341,0.1%,5.4%,2.24,81.424999,6.834348
Affiliate,12K,85.0%,97.0%,0.145616,-8.0%,-13.2%,6.05,140.808000,127.155379


In [18]:
# 
# 2) Compute ROI
df_roi_section = (df_roi, end_date, pp_end_date, cwly_date, pwly_date)

df_roi_section


(     week_ending channel_group       channel   contribution           cost  \
 0     2024-01-06           SEM     SEM Brand  350375.715323   57057.418927   
 1     2024-01-06     Affiliate            CJ  130388.713228   49504.710000   
 2     2024-01-06      Shop PPC    CLICKTRIPZ    5356.142557    3289.730000   
 3     2024-01-06      Shop PPC  CHEAPFLIGHTS  196020.059530  158197.808000   
 4     2024-01-06           SEM      SEM Core  712666.550000  745402.240000   
 ...          ...           ...           ...            ...            ...   
 1001  2027-01-23      Shop PPC        Criteo       0.000000       0.000000   
 1002  2027-01-30      Shop PPC           GDN       0.000000       0.000000   
 1003  2027-01-30      Shop PPC        Criteo       0.000000       0.000000   
 1004  2027-02-06      Shop PPC        Criteo       0.000000       0.000000   
 1005  2027-02-06      Shop PPC           GDN       0.000000       0.000000   
 
            roi  
 0     6.140756  
 1     2.63386

In [19]:
# 3) Build the presentation table
df_roi_v = create_roi_table(df_roi_section,df_dau_converison,end_date, pp_end_date, cwly_date, pwly_date,format_number)
df_roi_v['Conversion'] =format_percentage(df_roi_v['Conversion']*100)
df_roi_v

df_conversion_cy=df_roi_v.iloc[-1, 3]

df_conversion_cwly=df_dau_converison['conversion_cwly']['Total']

df_conversion_yoy = df_roi_v.iloc[-1, 4]

df_conversion_yoypw= df_roi_v.iloc[-1, 5]
df_roi_v

TypeError: create_roi_table() takes 3 positional arguments but 7 were given

## Parity data

In [20]:
# import importlib
from parity_table import create_parity_table
import parity_table

importlib.reload(parity_table)

df_parity = create_parity_table(
        format_percentage,
        df_direct_parity,
        # round_to_nearest_10,
        df_meta_parity)
df_parity

,Actual_pcln,YoY_pcln,YoY PW_pcln,Actual_exp,YoY_exp,YoY PW_exp
Direct vs Expedia,94.3%,1.2%,1.0%,NaN,NaN,NaN
Kayak Placement,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
Skyscanner Placement,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%


## Deal Share

In [21]:
deal_ticket_actual_cy=df_deal_share[(df_deal_share['wk_ending']==end_date)].groupby('deal_vs_non_deal')['totalTkts'].sum()
deal_ticket_actual_cy=deal_ticket_actual_cy[0]/(deal_ticket_actual_cy[0]+deal_ticket_actual_cy[1])
# deal_ticket_actual_cy=format_percentage(deal_ticket_actual_cy*100)

deal_ticket_actual_cy

deal_ticket_actual_pw=df_deal_share[(df_deal_share['wk_ending']==pp_end_date)].groupby('deal_vs_non_deal')['totalTkts'].sum()
deal_ticket_actual_pw=deal_ticket_actual_pw[0]/(deal_ticket_actual_pw[0]+deal_ticket_actual_pw[1])
# deal_ticket_actual_pw=format_percentage(deal_ticket_actual_pw*100)
deal_ticket_actual_pw

deal_ticket_actual_cwly=df_deal_share[(df_deal_share['wk_ending']==cwly_date)].groupby('deal_vs_non_deal')['totalTkts'].sum()
deal_ticket_actual_cwly=deal_ticket_actual_cwly[0]/(deal_ticket_actual_cwly[0]+deal_ticket_actual_cwly[1])
# deal_ticket_actual_cwly=format_percentage(deal_ticket_actual_cwly*100)
deal_ticket_actual_cwly

deal_ticket_actual_pwly=df_deal_share[(df_deal_share['wk_ending']==pwly_date)].groupby('deal_vs_non_deal')['totalTkts'].sum()
deal_ticket_actual_pwly=deal_ticket_actual_pwly[0]/(deal_ticket_actual_pwly[0]+deal_ticket_actual_pwly[1])
# deal_ticket_actual_pwly=format_percentage(deal_ticket_actual_pwly*100)
deal_ticket_actual_cy,deal_ticket_actual_cwly,deal_ticket_actual_pw,deal_ticket_actual_pwly

/var/folders/k6/ygnww5yd0rx_9c55x7xb7xs00000gn/T/ipykernel_5384/404268113.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  deal_ticket_actual_cy=deal_ticket_actual_cy[0]/(deal_ticket_actual_cy[0]+deal_ticket_actual_cy[1])
/var/folders/k6/ygnww5yd0rx_9c55x7xb7xs00000gn/T/ipykernel_5384/404268113.py:8: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  deal_ticket_actual_pw=deal_ticket_actual_pw[0]/(deal_ticket_actual_pw[0]+deal_ticket_actual_pw[1])
/var/folders/k6/ygnww5yd0rx_9c55x7xb7xs00000gn/T/ipykernel_5384/404268113.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future versi

(np.float64(0.2888528312130597),
 np.float64(0.3353769697575107),
 np.float64(0.30079167268321166),
 np.float64(0.3397743003502236))

## MIDT

In [21]:

# df_midt_data= pd.DataFrame()
# #Calulate actual
# a=df_midt[(df_midt['wk_ending']==end_date)].groupby(['agency'])['Tickets'].sum()['PRICELINE']
# b=df_midt[(df_midt['wk_ending']==end_date)]['Tickets'].sum()
# # Calculate the percentage
# Actual = ((a * 100) / b)


# #Calulate cwly_date
# a=df_midt[(df_midt['wk_ending']==cwly_date)].groupby(['agency'])['Tickets'].sum()['PRICELINE']
# b=df_midt[(df_midt['wk_ending']==cwly_date)]['Tickets'].sum()
# # Calculate the percentage
# Actual_cwly = ((a * 100) / b)

# #Calulate PWly_date
# a=df_midt[(df_midt['wk_ending']==pp_end_date)].groupby(['agency'])['Tickets'].sum()['PRICELINE']
# b=df_midt[(df_midt['wk_ending']==pp_end_date)]['Tickets'].sum()
# # Calculate the percentage
# actual_PW = ((a * 100) / b)

# #Calulate PWly_date
# a=df_midt[(df_midt['wk_ending']==PWly_date)].groupby(['agency'])['Tickets'].sum()['PRICELINE']
# b=df_midt[(df_midt['wk_ending']==PWly_date)]['Tickets'].sum()
# # Calculate the percentage
# actual_PWly = ((a * 100) / b)


# # Create a new DataFrame with the calculated 'Actual' column
# df_midt_data = pd.DataFrame()
# df_midt_data['Actual']=[format_percentage(Actual)]
# df_midt_data['YoY (bps)']=[round_to_nearest_10((Actual-Actual_cwly)*100)]
# df_midt_data['YoY PW(bps)']=[round_to_nearest_10((actual_PW-actual_PWly)*100)]
# df_midt_data
# # df_midt_data['Actual']

## SEM

In [22]:
df_sem=df_sem[(df_sem['DOMAIN']=='You')|(df_sem['DOMAIN']=='expedia.com')]
df_sem
# Calculate actuals for current, cwly, pp, and pwly dates
def compute_rate_ratio(df, date):
    grouped = df[df['wk_ending'] == date].groupby('DOMAIN')[['ABS_TOP_PAGE_RATE_IMPRESSIONS', 'TOTAL_IMPRESSIONS']].sum()
    grouped['RATE_RATIO'] = grouped['ABS_TOP_PAGE_RATE_IMPRESSIONS'] / grouped['TOTAL_IMPRESSIONS']
    return grouped['RATE_RATIO'] 

# Compute each set
actual = compute_rate_ratio(df_sem, end_date)
actual_cwly = compute_rate_ratio(df_sem, cwly_date)
actual_pw = compute_rate_ratio(df_sem, pp_end_date)
actual_pwly = compute_rate_ratio(df_sem, pwly_date)

df_sem_data = pd.DataFrame({
    'Actual': (actual*100).apply(format_percentage),
    'YoY': format_percentage(((actual/actual_cwly)-1) * 100),
    'YoY PW': format_percentage(((actual_pw/actual_pwly)-1) * 100)
})

df_sem_data.reset_index(inplace=True)  # optional, for clarity if you need DOMAIN as a column
df_sem_data


,DOMAIN,Actual,YoY,YoY PW
0,You,4.9%,8.3%,16.3%
1,expedia.com,13.5%,-22.4%,-10.2%


# Summary Table

## YTD

In [23]:
# Ensure the module is installed or available
import importlib
import summary_table_actual  # Import the module

# Reload the module to ensure the latest version is used
importlib.reload(summary_table_actual)

# Import the required function
from summary_table_actual import create_finance_number

# Call the function to create the finance number
finance_number = create_finance_number(
    df_weekly,
    df_gds_incentive,
    end_date,
    pp_end_date,
    begin_of_current_year,
    begin_of_last_year,
    ytd_last_year,
    cwly_date,
    pwly_date,
    format_percentage,
    format_number,
)
finance_number

,Measure,CW,PW,Reporting Week,Previous Week,CY,LY,YTD
0,Net Tickets,183538.0,181315.0,0.6%,2.3%,832549.0,822325.0,1.2%
1,Gross Tickets,197099.0,195329.0,0.4%,2.2%,894175.0,886214.0,0.9%
2,Net Revenue (net_contribution),1756955.0,1745022.0,-13.3%,-17.6%,8334904.0,9719182.0,-14.2%
3,Gross Revenue (gross_contribution),1909143.0,1906818.0,-14.8%,-18.6%,9054120.0,10749206.0,-15.8%
4,Normalized Net Tickets,158598.0,155764.0,1.8%,2.3%,718327.0,710103.0,1.2%
5,Normalized Gross Tickets,169758.0,167571.0,1.2%,1.9%,769487.0,766187.0,0.4%
6,Net Cont + Fee,1974921.0,1997446.0,-10.6%,-12.9%,9457352.0,10515782.0,-10.1%
7,Gross Cont + Fee,2142144.0,2175027.0,-12.1%,-14.1%,10249861.0,11610098.0,-11.7%
8,GDS Incentive,113513.0,104461.0,8.7%,0.0%,505915.0,878188.0,-42.4%
9,VCC Rebate (net_wex_fee),92796.0,87173.0,287.0%,214.0%,393902.0,0.0,


In [24]:
#current week finance data
#net tickets by scenario
plan_net_cw=df_finance[df_finance['wk_ending']==end_date.strftime('%Y-%m-%d')].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_cw=df_finance[df_finance['wk_ending']==end_date.strftime('%Y-%m-%d')].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_cw=df_finance[df_finance['wk_ending']==end_date.strftime('%Y-%m-%d')].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_cw=df_finance[df_finance['wk_ending']==end_date.strftime('%Y-%m-%d')].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)

#previous week finance data
#net tickets by scenario
plan_net_pw=df_finance[df_finance['wk_ending']==pp_end_date.strftime('%Y-%m-%d')].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_pw=df_finance[df_finance['wk_ending']==pp_end_date.strftime('%Y-%m-%d')].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_pw=df_finance[df_finance['wk_ending']==pp_end_date.strftime('%Y-%m-%d')].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_pw=df_finance[df_finance['wk_ending']==pp_end_date.strftime('%Y-%m-%d')].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)


#ytd week finance data
#net tickets by scenario
plan_net_ytd=df_finance[(df_finance['wk_ending']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')['net_units'].sum()
#gross tickets by scenario
plan_gr_ytd=df_finance[(df_finance['wk_ending']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')['gr_Units'].sum()
#net revenue by scenario
plan_grrev_ytd=df_finance[(df_finance['wk_ending']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
#net revenue by scenario
plan_netrev_ytd=df_finance[(df_finance['wk_ending']<=end_date.strftime('%Y-%m-%d'))
& (df_finance['trans_date']>=begin_of_current_year.strftime('%Y-%m-%d'))].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)



# #ytd last year finance data
# #net tickets by scenario
# plan_net_ytd_ly=df_finance[(df_finance['wk_ending']<=ytd_last_year.strftime('%Y-%m-%d'))
# & (df_finance['trans_date']>=begin_of_last_year.strftime('%Y-%m-%d'))].groupby('scenario')['net_units'].sum()
# #gross tickets by scenario
# plan_gr_ytd_ly=df_finance[(df_finance['wk_ending']<=ytd_last_year.strftime('%Y-%m-%d'))
# & (df_finance['trans_date']>=begin_of_last_year.strftime('%Y-%m-%d'))].groupby('scenario')['gr_Units'].sum()
# #net revenue by scenario
# plan_grrev_ytd_ly=df_finance[(df_finance['wk_ending']<=ytd_last_year.strftime('%Y-%m-%d'))
# & (df_finance['trans_date']>=begin_of_last_year.strftime('%Y-%m-%d'))].groupby('scenario')[['gr_cont_fee_GDS_Wex']].sum().round(0)
# #net revenue by scenario
# plan_netrev_ytd_ly=df_finance[(df_finance['wk_ending']<=ytd_last_year.strftime('%Y-%m-%d'))
# & (df_finance['trans_date']>=begin_of_last_year.strftime('%Y-%m-%d'))].groupby('scenario')[['net_cont_fee_GDS_Wex']].sum().round(0)


In [25]:
period_order = ["CW", "PW", "YTD", "YTD_LY"]

# revenue as Series (no squeeze)
plan_grrev_cw     = df_finance.loc[df_finance["wk_ending"] == end_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_pw     = df_finance.loc[df_finance["wk_ending"] == pp_end_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_ytd    = df_finance.loc[
    (df_finance["wk_ending"] <= end_date.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_current_year.strftime("%Y-%m-%d"))
].groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_grrev_ytd_ly = df_finance.loc[
    (df_finance["wk_ending"] <= ytd_last_year.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_last_year.strftime("%Y-%m-%d"))
].groupby("scenario")["gr_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_cw     = df_finance.loc[df_finance["wk_ending"] == end_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_pw     = df_finance.loc[df_finance["wk_ending"] == pp_end_date.strftime("%Y-%m-%d")] \
    .groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_ytd    = df_finance.loc[
    (df_finance["wk_ending"] <= end_date.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_current_year.strftime("%Y-%m-%d"))
].groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

plan_netrev_ytd_ly = df_finance.loc[
    (df_finance["wk_ending"] <= ytd_last_year.strftime("%Y-%m-%d")) &
    (df_finance["trans_date"] >= begin_of_last_year.strftime("%Y-%m-%d"))
].groupby("scenario")["net_cont_fee_GDS_Wex"].sum().round(0)

# 1) MultiIndex columns (metric, period)
plan_metrics_mi = pd.concat(
    {
        ("net_tkts", "CW"): plan_net_cw,
        ("net_tkts", "PW"): plan_net_pw,
        ("net_tkts", "YTD"): plan_net_ytd,
        # ("net_tkts", "YTD_LY"): plan_net_ytd_ly,

        ("gr_tkts", "CW"): plan_gr_cw,
        ("gr_tkts", "PW"): plan_gr_pw,
        ("gr_tkts", "YTD"): plan_gr_ytd,
        # ("gr_tkts", "YTD_LY"): plan_gr_ytd_ly,

        ("gr_rev", "CW"): plan_grrev_cw,
        ("gr_rev", "PW"): plan_grrev_pw,
        ("gr_rev", "YTD"): plan_grrev_ytd,
        # ("gr_rev", "YTD_LY"): plan_grrev_ytd_ly,

        ("net_rev", "CW"): plan_netrev_cw,
        ("net_rev", "PW"): plan_netrev_pw,
        ("net_rev", "YTD"): plan_netrev_ytd,
        # ("net_rev", "YTD_LY"): plan_netrev_ytd_ly,
    },
    axis=1
)
plan_metrics_mi.index.name = "scenario"

# 2) reshape to "period columns"
plan_metrics_period_cols = (
    plan_metrics_mi
      .stack(0)                # stack metric level -> rows
      .reset_index()
      .rename(columns={"level_1": "metric"})
)

# optional formatting
if callable(format_number):
    for c in period_order:
        if c in plan_metrics_period_cols.columns:
            plan_metrics_period_cols[c] = plan_metrics_period_cols[c].round(0)

keep = ["scenario", "metric"] + [c for c in period_order if c in plan_metrics_period_cols.columns]
plan_metrics_period = plan_metrics_period_cols[keep]

# filter PLAN
plan_metrics_period_plan = plan_metrics_period.loc[plan_metrics_period["scenario"].eq("PLAN")]
plan_metrics_period_plan

/var/folders/k6/ygnww5yd0rx_9c55x7xb7xs00000gn/T/ipykernel_5384/784099965.py:65: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  plan_metrics_mi


,scenario,metric,CW,PW,YTD
0,PLAN,gr_rev,1991872.0,1954948.0,8909896.0
1,PLAN,gr_tkts,200075.0,192753.0,883057.0
2,PLAN,net_rev,1879070.0,1845538.0,8423038.0
3,PLAN,net_tkts,187723.0,180837.0,830837.0


In [26]:
def build_vs_plan(finance_number: pd.DataFrame, plan_metrics_period: pd.DataFrame,format_percentage) -> pd.DataFrame:
    plan = (plan_metrics_period.loc[plan_metrics_period["scenario"].eq("PLAN"),
                                    ["metric", "CW", "PW", "YTD"]]
            .set_index("metric")
            .rename(columns={"CW": "CW_plan", "PW": "PW_plan", "YTD": "YTD_plan"}))

    actual = (finance_number[["Measure", "CW", "PW", "CY"]]
              .set_index("Measure"))

    measure_to_metric = {
        "Net Tickets": "net_tkts",
        "Gross Tickets": "gr_tkts",
        "Net Cont + Fee + Incentives + vcc rebate(Flight Only)": "net_rev",
        "Gross Cont + Fee + Incentives + vcc rebate(Flight Only)": "gr_rev",
    }

    tmp = actual.rename(index=measure_to_metric)
    tmp = tmp.join(plan, how="right")
    
    metric_order = list(measure_to_metric.values())
    tmp = tmp.reindex(metric_order)
    def vs_pct(a, p):
        p = p.replace(0, np.nan)
        return (a / p - 1) * 100

    out = pd.DataFrame(index=tmp.index)
    out["Reporting Week (vs Plan)"] = vs_pct(tmp["CW"], tmp["CW_plan"])
    out["Previous Week (vs Plan)"]  = vs_pct(tmp["PW"], tmp["PW_plan"])
    out["YTD (vs Plan)"]            = vs_pct(tmp["CY"], tmp["YTD_plan"])  # CY = YTD actual

    metric_to_measure = {v: k for k, v in measure_to_metric.items()}
    out.index = out.index.map(metric_to_measure)

    for c in ["Reporting Week (vs Plan)", "Previous Week (vs Plan)", "YTD (vs Plan)"]:
        out[c] = out[c].apply(format_percentage)

    return out


In [27]:
df_vs_plan = build_vs_plan(finance_number, plan_metrics_period, format_percentage)
df_vs_plan

,Reporting Week (vs Plan),Previous Week (vs Plan),YTD (vs Plan)
metric,,,
Net Tickets,-2.2%,0.3%,0.2%
Gross Tickets,-1.5%,1.3%,1.3%
Net Cont + Fee + Incentives + vcc rebate(Flight Only),-3.9%,-2.8%,0.5%
Gross Cont + Fee + Incentives + vcc rebate(Flight Only),-2.4%,-0.2%,2.6%


In [28]:
finance_number
wanted = [
    "Net Tickets",
    "Gross Tickets",
    "Net Cont + Fee + Incentives + vcc rebate(Flight Only)",
    "Gross Cont + Fee + Incentives + vcc rebate(Flight Only)",
    "Normalized Net Tickets",
    "Normalized Gross Tickets",
]

finance_output = finance_number.loc[
    finance_number["Measure"].isin(wanted),
    ["Measure", "CW", "Reporting Week", "Previous Week", "YTD"]
].reset_index(drop=True)

# --- formatting ---
# numeric columns
for c in ["CW"]:
    if c in finance_output.columns:
        finance_output[c] = finance_output[c].apply(format_number)
        
finance_output=finance_output.set_index('Measure')
finance_output=finance_output.rename(columns={"CW": "Actual"})
finance_output=finance_output.reindex(wanted).fillna('')
finance_output


,Actual,Reporting Week,Previous Week,YTD
Measure,,,,
Net Tickets,184K,0.6%,2.3%,1.2%
Gross Tickets,197K,0.4%,2.2%,0.9%
Net Cont + Fee + Incentives + vcc rebate(Flight Only),1.8M,-9.6%,-13.3%,-11.5%
Gross Cont + Fee + Incentives + vcc rebate(Flight Only),1.9M,-11.1%,-14.2%,-12.9%
Normalized Net Tickets,159K,1.8%,2.3%,1.2%
Normalized Gross Tickets,170K,1.2%,1.9%,0.4%


## Last year and plan section

## TSA

In [29]:
tsa_cy=df_tsa[df_tsa['wk_ending']==end_date]['tsa_passengers'].sum()
pcln_cy=df_tsa[df_tsa['wk_ending']==end_date]['pcln_passengers'].sum()
tsa_actual_cy=(pcln_cy*100/tsa_cy)


tsa_pw=df_tsa[df_tsa['wk_ending']==pp_end_date]['tsa_passengers'].sum()
pcln_pw=df_tsa[df_tsa['wk_ending']==pp_end_date]['pcln_passengers'].sum()
tsa_actual_pw=(pcln_pw*100/tsa_pw)


tsa_cwly=df_tsa[df_tsa['wk_ending']==cwly_date]['tsa_passengers'].sum()
pcln_cwly=df_tsa[df_tsa['wk_ending']==cwly_date]['pcln_passengers'].sum()
tsa_actual_cwly=(pcln_cwly*100/tsa_cwly)



tsa_pwly=df_tsa[df_tsa['wk_ending']==pwly_date]['tsa_passengers'].sum()
pcln_pwly=df_tsa[df_tsa['wk_ending']==pwly_date]['pcln_passengers'].sum()
tsa_actual_pwly=(pcln_pwly*100/tsa_pwly)


tsa_ytd=df_tsa[(df_tsa['date'] <=end_date)&(df_tsa['date']>=begin_of_current_year)]['tsa_passengers'].sum()
pcln_ytd=df_tsa[(df_tsa['date']<=end_date)& (df_tsa['date']>=begin_of_current_year)]['pcln_passengers'].sum()
tsa_actual_ytd=round((pcln_ytd*100/tsa_ytd),2)


tsa_ytd_ly=df_tsa[(df_tsa['date'] <=end_date)&(df_tsa['date']>=begin_of_last_year)]['tsa_passengers'].sum()
pcln_ytd_ly=df_tsa[(df_tsa['date']<=end_date)& (df_tsa['date']>=begin_of_last_year)]['pcln_passengers'].sum()
tsa_actual_ytd_ly=round((pcln_ytd_ly*100/tsa_ytd_ly),2)

print("TSA CY",tsa_cy,"PCLN CY",pcln_cy,"%TSA market share",tsa_actual_cy)
print("TSA LY",tsa_cwly,"PCLN LY",pcln_cwly,"%TSAmarket share",tsa_actual_cwly)
print("TSA PW",tsa_pw,"PCLN PW",pcln_pw,"%TSAmarket share",tsa_actual_pw)
print("TSA PWLY",tsa_pwly,"PCLN PWly",pcln_pwly,"%TSAmarket share",tsa_actual_pwly)
print("TSA YTD CY",tsa_ytd,"PCLN YTD CY",pcln_ytd,"%TSAmarket share CY YTD ",tsa_actual_ytd)
print("TSA YTD LY",tsa_ytd_ly,"PCLN YTD LY",pcln_cwly,"%TSA market share LY YTD ",tsa_actual_ytd_ly)

TSA CY 12991706 PCLN CY 181589.0 %TSA market share 1.3977302134146201
TSA LY 14066455 PCLN LY 180844.0 %TSAmarket share 1.285640198614363
TSA PW 14638756 PCLN PW 198030.0 %TSAmarket share 1.3527788836701697
TSA PWLY 14251653 PCLN PWly 194262.0 %TSAmarket share 1.3630839875206056
TSA YTD CY 65860911 PCLN YTD CY 927310.0 %TSAmarket share CY YTD  1.41
TSA YTD LY 972596887 PCLN YTD LY 180844.0 %TSA market share LY YTD  1.41


In [30]:
# df_summary=df_summary.reset_index(names=['Metric'])


df_summary= pd.DataFrame()
df_summary.loc['DAU','Actual']=df_roi_v.iloc[:, 0].loc['Total']
df_summary.loc['DAU','Reporting Week']=df_roi_v.iloc[:, 1].loc['Total']
df_summary.loc['DAU','Previous Week']=df_roi_v.iloc[:, 2].loc['Total']
df_summary.loc['DAU','YTD']=df_dau_converison.loc[:, "DAU YoY_YTD"].loc['Total']


df_summary.loc['TSA','Actual']=format_percentage_2(tsa_actual_cy)
df_summary.loc['TSA','Reporting Week']=format_percentage(((tsa_actual_cy/tsa_actual_cwly)-1)*100)
df_summary.loc['TSA','Previous Week']=format_percentage((tsa_actual_pw/tsa_actual_pwly-1)*100)
df_summary.loc['TSA','YTD']=format_percentage((tsa_actual_ytd/tsa_actual_ytd_ly-1)*100)

df_summary=df_summary.fillna('')

df_summary

,Actual,Reporting Week,Previous Week,YTD
DAU,2.0M,6.7%,8.3%,-3.0%
TSA,1.40%,8.7%,-0.8%,0.0%


In [31]:
final_summary = pd.concat([finance_output,df_summary])
final_summary = final_summary.join(df_vs_plan, how="left").fillna('')
final_summary

,Actual,Reporting Week,Previous Week,YTD,Reporting Week (vs Plan),Previous Week (vs Plan),YTD (vs Plan)
Net Tickets,184K,0.6%,2.3%,1.2%,-2.2%,0.3%,0.2%
Gross Tickets,197K,0.4%,2.2%,0.9%,-1.5%,1.3%,1.3%
Net Cont + Fee + Incentives + vcc rebate(Flight Only),1.8M,-9.6%,-13.3%,-11.5%,-3.9%,-2.8%,0.5%
Gross Cont + Fee + Incentives + vcc rebate(Flight Only),1.9M,-11.1%,-14.2%,-12.9%,-2.4%,-0.2%,2.6%
Normalized Net Tickets,159K,1.8%,2.3%,1.2%,,,
Normalized Gross Tickets,170K,1.2%,1.9%,0.4%,,,
DAU,2.0M,6.7%,8.3%,-3.0%,,,
TSA,1.40%,8.7%,-0.8%,0.0%,,,


## TSA web scraping 

In [32]:

# from tsa_table import create_tsa_web_table
# df_tsa_web=create_tsa_web_table()
# df_tsa_web=df_tsa_web.dropna()

# import tsa_table
# import importlib

# importlib.reload(tsa_table)
# from tsa_table import create_df_tsa_table

# df_tsa=create_df_tsa_table(tsa_data,df_tsa_web,format_percentage,end_date,pp_end_date,cwly_date,pwly_date)
# df_tsa


## Upsell

In [32]:
# df_upsell
upsell_cy=df_upsell[(df_upsell['week_ending']==end_date)]['upsell_tickets'].sum()
select_cy=df_upsell[(df_upsell['week_ending']==end_date)]['Selected_tickets'].sum()
upsell_rate_cy=upsell_cy/(upsell_cy+select_cy)

upsell_pw=df_upsell[(df_upsell['week_ending']==pp_end_date)]['upsell_tickets'].sum()
select_pw=df_upsell[(df_upsell['week_ending']==pp_end_date)]['Selected_tickets'].sum()
upsell_rate_pw=upsell_pw/(upsell_pw+select_pw)

upsell_cwly=df_upsell[(df_upsell['week_ending']==cwly_date)]['upsell_tickets'].sum()
select_cwly=df_upsell[(df_upsell['week_ending']==cwly_date)]['Selected_tickets'].sum()
upsell_rate_cwly=upsell_cwly/(upsell_cwly+select_cwly)

upsell_pwly=df_upsell[(df_upsell['week_ending']==pwly_date)]['upsell_tickets'].sum()
select_pwly=df_upsell[(df_upsell['week_ending']==pwly_date)]['Selected_tickets'].sum()
upsell_rate_pwly=upsell_pwly/(upsell_pwly+select_pwly)

upsell_rate_cy,upsell_rate_pw,upsell_rate_cwly,upsell_rate_pwly

(np.float64(0.1899270863362466),
 np.float64(0.19125027274710887),
 np.float64(0.11481739257848272),
 np.float64(0.11772646267015509))

In [33]:
def calculate_upsell_rate(df, date_column, target_date):
    upsell = df[df[date_column] == target_date]['upsell_tickets'].sum()
    selected = df[df[date_column] == target_date]['Selected_tickets'].sum()
    return upsell / (upsell + selected) if (upsell + selected) > 0 else 0

# Calculate upsell rates for different dates
upsell_rate_cy = calculate_upsell_rate(df_upsell, 'week_ending', end_date)
upsell_rate_pw = calculate_upsell_rate(df_upsell, 'week_ending', pp_end_date)
upsell_rate_cwly = calculate_upsell_rate(df_upsell, 'week_ending', cwly_date)
upsell_rate_pwly = calculate_upsell_rate(df_upsell, 'week_ending', pwly_date)

# Output the results
upsell_rate_cy, upsell_rate_pw, upsell_rate_cwly, upsell_rate_pwly

(np.float64(0.1899270863362466),
 np.float64(0.19125027274710887),
 np.float64(0.11481739257848272),
 np.float64(0.11772646267015509))

## Bookability

In [34]:
df_bookability
# df_upsell
bookability_cy=df_bookability[(df_bookability['week_ending']==end_date)]['success_rate'].iloc[0].tolist()

bookability_pw=df_bookability[(df_bookability['week_ending']==pp_end_date)]['success_rate'].iloc[0].tolist()

bookability_cwly=df_bookability[(df_bookability['week_ending']==cwly_date)]['success_rate'].iloc[0].tolist()

bookability_pwly=df_bookability[(df_bookability['week_ending']==pwly_date)]['success_rate'].iloc[0].tolist()

bookability_cy ,bookability_pw,bookability_cwly,bookability_pwly

(0.7445131571064352,
 0.7443701005351395,
 0.7585408795078238,
 0.7493158625524057)

## Booking Rate

In [35]:
net_ticket_cy = df_pricelince.loc[df_pricelince['wk_ending'] == end_date, 'net_tkts_cy'].sum()
gr_ticket_cy= df_pricelince.loc[df_pricelince['wk_ending'] == end_date, 'gr_tkts_cy'].sum()
booking_rate_cy=net_ticket_cy/gr_ticket_cy

net_ticket_pw = df_pricelince.loc[df_pricelince['wk_ending'] == pp_end_date, 'net_tkts_cy'].sum()
gr_ticket_pw= df_pricelince.loc[df_pricelince['wk_ending'] == pp_end_date, 'gr_tkts_cy'].sum()
booking_rate_pw=net_ticket_pw/gr_ticket_pw

net_ticket_cwly = df_pricelince.loc[df_pricelince['wk_ending'] == end_date, 'net_tkts_ly'].sum()
gr_ticket_cwly= df_pricelince.loc[df_pricelince['wk_ending'] == end_date, 'gr_tkts_ly'].sum()
booking_rate_cwly=net_ticket_cwly/gr_ticket_cwly


net_ticket_pwly = df_pricelince.loc[df_pricelince['wk_ending'] == pp_end_date, 'net_tkts_ly'].sum()
gr_ticket_pwly= df_pricelince.loc[df_pricelince['wk_ending'] == pp_end_date, 'gr_tkts_ly'].sum()
booking_rate_pwly=net_ticket_pwly/gr_ticket_pwly

booking_rate_cy

np.float64(0.9311970126687603)

## Ticket/Order

In [36]:
net_ticket_cy = df_pricelince.loc[df_pricelince['wk_ending'] == end_date, 'net_tkts_cy'].sum()
net_order_cy= df_pricelince.loc[df_pricelince['wk_ending'] == end_date, 'net_orders_cy'].sum()
tickets_order_cy=net_ticket_cy/net_order_cy

net_ticket_pw = df_pricelince.loc[df_pricelince['wk_ending'] == pp_end_date, 'net_tkts_cy'].sum()
net_order_pw= df_pricelince.loc[df_pricelince['wk_ending'] == pp_end_date, 'net_orders_cy'].sum()
tickets_order_pw=net_ticket_pw/net_order_pw

net_ticket_cwly = df_pricelince.loc[df_pricelince['wk_ending'] == end_date, 'net_tkts_ly'].sum()
net_order_cwly= df_pricelince.loc[df_pricelince['wk_ending'] == end_date, 'net_orders_ly'].sum()
tickets_order_cwly=net_ticket_cwly/net_order_cwly

net_ticket_pwly = df_pricelince.loc[df_pricelince['wk_ending'] == pp_end_date, 'net_tkts_ly'].sum()
net_order_pwly= df_pricelince.loc[df_pricelince['wk_ending'] == pp_end_date, 'net_orders_ly'].sum()
tickets_order_pwly=net_ticket_pwly/net_order_pwly

tickets_order_cy



np.float64(1.4997262646979514)

In [37]:
# Define a generic helper function to calculate metrics
def calculate_metric(df, date_column, target_date, numerator_column, denominator_column, operation):
    numerator = df.loc[df[date_column] == target_date, numerator_column].sum()
    denominator = df.loc[df[date_column] == target_date, denominator_column].sum()
    if denominator > 0:
        return operation(numerator, denominator)
    return 0

# Define specific operations for each metric
upsell_rate_operation = lambda upsell, selected: upsell / (upsell + selected)
booking_rate_operation = lambda net, gr: net / gr
tickets_per_order_operation = lambda tickets, orders: tickets / orders

# Calculate upsell rates
upsell_rate_cy = calculate_metric(df_upsell, 'week_ending', end_date, 'upsell_tickets', 'Selected_tickets', upsell_rate_operation)
upsell_rate_pw = calculate_metric(df_upsell, 'week_ending', pp_end_date, 'upsell_tickets', 'Selected_tickets', upsell_rate_operation)
upsell_rate_cwly = calculate_metric(df_upsell, 'week_ending', cwly_date, 'upsell_tickets', 'Selected_tickets', upsell_rate_operation)
upsell_rate_pwly = calculate_metric(df_upsell, 'week_ending', pwly_date, 'upsell_tickets', 'Selected_tickets', upsell_rate_operation)

# Calculate booking rates
booking_rate_cy = calculate_metric(df_pricelince, 'wk_ending', end_date, 'net_tkts_cy', 'gr_tkts_cy', booking_rate_operation)
booking_rate_pw = calculate_metric(df_pricelince, 'wk_ending', pp_end_date, 'net_tkts_cy', 'gr_tkts_cy', booking_rate_operation)
booking_rate_cwly = calculate_metric(df_pricelince, 'wk_ending', end_date, 'net_tkts_ly', 'gr_tkts_ly', booking_rate_operation)
booking_rate_pwly = calculate_metric(df_pricelince, 'wk_ending', pp_end_date, 'net_tkts_ly', 'gr_tkts_ly', booking_rate_operation)

# Calculate tickets per order
tickets_order_cy = calculate_metric(df_pricelince, 'wk_ending', end_date, 'net_tkts_cy', 'net_orders_cy', tickets_per_order_operation)
tickets_order_pw = calculate_metric(df_pricelince, 'wk_ending', pp_end_date, 'net_tkts_cy', 'net_orders_cy', tickets_per_order_operation)
tickets_order_cwly = calculate_metric(df_pricelince, 'wk_ending', end_date, 'net_tkts_ly', 'net_orders_ly', tickets_per_order_operation)
tickets_order_pwly = calculate_metric(df_pricelince, 'wk_ending', pp_end_date, 'net_tkts_ly', 'net_orders_ly', tickets_per_order_operation)

# Output the results
(upsell_rate_cy, upsell_rate_pw, upsell_rate_cwly, upsell_rate_pwly,
 booking_rate_cy, booking_rate_pw, booking_rate_cwly, booking_rate_pwly,
 tickets_order_cy, tickets_order_pw, tickets_order_cwly, tickets_order_pwly)

(np.float64(0.1899270863362466),
 np.float64(0.19125027274710887),
 np.float64(0.11481739257848272),
 np.float64(0.11772646267015509),
 np.float64(0.9311970126687603),
 np.float64(0.9282543810698872),
 np.float64(0.9300404602480611),
 np.float64(0.9276923640665651),
 np.float64(1.4997262646979514),
 np.float64(1.5112353931554119),
 np.float64(1.4772160709660709),
 np.float64(1.490291384602447))

## Others Table

In [39]:
df_others = pd.DataFrame()

#midt
# df_others.loc['MIDT','Actual_pcln']=df_midt_data.loc[0,'Actual']
# df_others.loc['MIDT','YoY (bps)_pcln']=df_midt_data.loc[0,'YoY (bps)']
# df_others.loc['MIDT','YoY PW (bps)_pcln']=df_midt_data.loc[0,'YoY PW(bps)']

# sem
df_others.loc['SEM Impressions','Actual_pcln']=df_sem_data.loc[0,'Actual']
df_others.loc['SEM Impressions','YoY_pcln']=df_sem_data.loc[0,'YoY']
df_others.loc['SEM Impressions','YoY PW_pcln']=df_sem_data.loc[0,'YoY PW']

df_others.loc['SEM Impressions','Actual_exp']=df_sem_data.loc[1,'Actual']
df_others.loc['SEM Impressions','YoY_exp']=df_sem_data.loc[1,'YoY']
df_others.loc['SEM Impressions','YoY PW_exp']=df_sem_data.loc[1,'YoY PW']


# Parity
df_others = pd.concat([df_others, df_parity.iloc[0:]])

# US Outbound & Merchant
df_others.loc['US Outbound','Actual_pcln'] = df_share_business.loc['US Outbound','Actual']
df_others.loc['US Outbound','YoY_pcln'] = df_share_business.loc['US Outbound','YoY']
df_others.loc['US Outbound','YoY PW_pcln'] = df_share_business.loc['US Outbound','YoY_PW']

df_others.loc['Merchant Retail','Actual_pcln'] = df_share_business.loc['Retail (Disclosed)','Actual']
df_others.loc['Merchant Retail','YoY_pcln'] = df_share_business.loc['Retail (Disclosed)','YoY']
df_others.loc['Merchant Retail','YoY PW_pcln'] = df_share_business.loc['Retail (Disclosed)','YoY_PW']

df_others.loc['Merchant Total','Actual_pcln'] = df_share_business.loc['Total','Actual']
df_others.loc['Merchant Total','YoY_pcln'] = df_share_business.loc['Total','YoY']
df_others.loc['Merchant Total','YoY PW_pcln'] = df_share_business.loc['Total','YoY_PW']

# Deal Share
df_others.loc['Deal Share','Actual_pcln'] = format_percentage(deal_ticket_actual_cy * 100)
df_others.loc['Deal Share','YoY_pcln'] = format_percentage(((deal_ticket_actual_cy/deal_ticket_actual_cwly)-1) * 100)
df_others.loc['Deal Share','YoY PW_pcln'] = format_percentage(((deal_ticket_actual_pw/deal_ticket_actual_pwly)-1) * 100)


# Conversion
df_others.loc['Conversion','Actual_pcln'] =0
# (df_conversion_cy)
df_others.loc['Conversion','YoY_pcln'] = 0
# (df_conversion_yoy)
df_others.loc['Conversion','YoY PW_pcln'] =0
# (df_conversion_yoypw)


#bookbility
df_others.loc['Bookability','Actual_pcln']=format_percentage(bookability_cy*100)
df_others.loc['Bookability','YoY_pcln']= format_percentage(((bookability_cy /bookability_cwly)-1)*100)
df_others.loc['Bookability','YoY PW_pcln']=format_percentage(((bookability_pw/bookability_pwly)-1)*100)

# Upsell
df_others.loc['Upsell','Actual_pcln'] = format_percentage(upsell_rate_cy*100)
df_others.loc['Upsell','YoY_pcln'] = format_percentage(((upsell_rate_cy/ upsell_rate_cwly)-1)*100)
df_others.loc['Upsell','YoY PW_pcln'] = format_percentage(((upsell_rate_pw /upsell_rate_pwly)-1)*100)



#booking rate
df_others.loc['Booking rate','Actual_pcln']=format_percentage(booking_rate_cy*100)
df_others.loc['Booking rate','YoY_pcln']= format_percentage(((booking_rate_cy/booking_rate_cwly)-1)*100)
df_others.loc['Booking rate','YoY PW_pcln']=format_percentage(((booking_rate_pw/booking_rate_pwly)-1)*100)

#Tickets/Order
df_others.loc['Tickets/order','Actual_pcln']=round(tickets_order_cy,3)
df_others.loc['Tickets/order','YoY_pcln']= format_percentage(((tickets_order_cy /tickets_order_cwly)-1)*100)
df_others.loc['Tickets/order','YoY PW_pcln']=format_percentage(((tickets_order_pw /tickets_order_pwly)-1)*100)

df_others=df_others.fillna('')
df_others

,Actual_pcln,YoY_pcln,YoY PW_pcln,Actual_exp,YoY_exp,YoY PW_exp
SEM Impressions,4.9%,8.3%,16.3%,13.5%,-22.4%,-10.2%
Direct vs Expedia,94.3%,1.2%,1.0%,,,
Kayak Placement,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
Skyscanner Placement,0.0%,0.0%,0.0%,0.0%,0.0%,0.0%
US Outbound,9.8%,-3.6%,-5.4%,,,
Merchant Retail,15.0%,81.8%,67.9%,,,
Merchant Total,21.0%,21.8%,14.4%,,,
Deal Share,28.9%,-13.9%,-11.5%,,,
Conversion,0,0,0,,,
Bookability,74.5%,-1.8%,-0.7%,,,


In [ ]:
print('SEM Impressions:',deal_ticket_actual_cy,deal_ticket_actual_cwly,deal_ticket_actual_pw,deal_ticket_actual_pwly)
print('Bookability:',bookability_cy,bookability_cwly,bookability_pw,bookability_pwly)
print('Upsell:',upsell_rate_cy,upsell_rate_cwly,upsell_rate_pw,upsell_rate_pwly)
print('Deal_share:',deal_ticket_actual_cy,deal_ticket_actual_cwly,deal_ticket_actual_pw,deal_ticket_actual_pwly)

## Revenue Table

In [76]:
import kpi_rev

importlib.reload(kpi_rev)

from kpi_rev import calculate_business_metrics

# Calculate business metrics
df_business_rev = calculate_business_metrics(df_pricelince,end_date, pp_end_date)

from kpi_rev import calculate_carrier_metrics
# Calculate carrier metrics
df_carrier_rev = calculate_carrier_metrics(df_pricelince_b2c_standalone, end_date, pp_end_date)

from kpi_rev import calculate_channel_metrics
# Calculate channel metrics
df_channel_rev = calculate_channel_metrics(df_pricelince_b2c_standalone, end_date, pp_end_date)

from kpi_rev import calculate_source_metrics
# Calculate source metrics
df_source_rev = calculate_source_metrics(df_pricelince_b2c_standalone, end_date, pp_end_date)

from kpi_rev import calculate_brand_metrics
# Calculate brand metrics
df_brand_rev = calculate_brand_metrics(df_weekly, end_date, pp_end_date)

In [ ]:

import config_table
import importlib

importlib.reload(config_table)
from docx import Document
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from docx.shared import Pt

from config_table import (
    clear_document,
    set_font,
    create_word_table,
    create_summary_table,
    create_others_table,
    # create_dau_table,
    create_roi_table
)


# Data preparation and configurations
carrier_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/Retail.jpg', 'title': 'Retail'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/Express.jpg', 'title': 'Express Deals'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

business_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/Standalone.jpg', 'title': 'Standalone'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/Package.jpg', 'title': 'Package'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

channel_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/App.jpg', 'title': 'App'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/MWeb Desktop.jpg', 'title': 'Web'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

source_logos = [
    {'start_col': 1, 'end_col': 3, 'path': '../Screenshots/weeklynote/Published.jpg', 'title': 'Published'},
    {'start_col': 4, 'end_col': 6, 'path': '../Screenshots/weeklynote/Private.jpg', 'title': 'Private'},
    {'start_col': 7, 'end_col': 9, 'path': '../Screenshots/weeklynote/Total.jpg', 'title': 'Total'}
]

def customize_bullet_style(paragraph, font_size):
    """
    Customize bullet style in a paragraph using XML manipulation.
    """
    # Access the paragraph's properties
    pPr = paragraph._element.get_or_add_pPr()
    numPr = OxmlElement('w:numPr')
    ilvl = OxmlElement('w:ilvl')
    ilvl.set(qn('w:val'), '0')  # Set indentation level
    numId = OxmlElement('w:numId')
    numId.set(qn('w:val'), '1')  # Set numbering ID

    numPr.append(ilvl)
    numPr.append(numId)
    pPr.append(numPr)

    # Modify font properties
    for run in paragraph.runs:
        run.font.name = "Montserrat"
        run.font.size = font_size

# Create a Word Document
word_document = Document()

# Clear the document (if necessary)
clear_document(word_document)

# Add Title
word_document.add_paragraph()
word_document.add_paragraph(f'Summary')
set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(18), bold=True)
word_document.add_paragraph('\n')

# Create Summary Table
create_summary_table(word_document,final_summary)

# Add General Notes
summary_notes = [
    'All Priceline tickets (includes B2B, Package, Express Deals, Phone Sales)',
    'Normalized tickets are counted the same as tickets, except split tickets count as 1 instead of 2',
    'Refunds are assigned to refund date',
    'Revenue is contribution w/fee + GDS incentives + VCC rebates.  Does not include package or phone sales.',
    'Daily Active Users: engaged customers in GA4',
    'TSA footprint: travel date; numerator counts all slices (OW is 1 slice; RT is 2 slices) where the first segment is either US domestic or US outbound (Priceline-only); denominator includes all people passing through TSA screening machines'
]
for note in summary_notes:
    word_document.add_paragraph(note, style='List Bullet')
    set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(7), bold=False)
    customize_bullet_style(word_document.paragraphs[-1], font_size=Pt(7))

# Add Tables for Business, Carrier, Channel, and Source
tables = [
    (df_business, "Business", business_logos, [
        'All Priceline tickets (includes Express Deals and Phone Sales, not normalized)'
    ]),
    (df_carrier, "Carrier", carrier_logos, [
        'Priceline (including phone sales), B2C, Standalone (not normalized)'
    ]),
    (df_channel, "Channel", channel_logos, [
        'Priceline (including phone sales), B2C, Standalone (includes Express Deals, not normalized)'
    ]),
    (df_source, "Source", source_logos, [
        'Priceline (including phone sales), B2C, Standalone (includes Express Deals, not normalized)',
        'Indirect Connect = GDS + Aggregators + Consolidators + NDC-X'
    ])
]
for df, title, logos, notes in tables:
    if df is None:
        continue

   # Add specific custom notes for each table
    if title == "Business":
        custom_note = "Total Business - Detail"
    elif title == "Carrier":
        custom_note = "Priceline B2C Standalone - Detail"
    else:
        custom_note = " "

    # Add the custom note before the table
    word_document.add_paragraph(custom_note)
    set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(18), bold=True)
    word_document.add_paragraph()

    create_word_table(df, title, logos, word_document)

    # word_document.add_paragraph()

    for note in notes:
        word_document.add_paragraph(note, style='List Bullet')
        # word_document.add_paragraph('\n')
        set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(7), bold=False)

word_document.add_paragraph('\n')
word_document.add_paragraph('Other Metrics')

set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(18), bold=True)

create_others_table(word_document, df_others)

others_notes = [
    'ARC footprint: OTA only, US POS, does not include NK/F9/SY',
    'SEM impression share: percentage of eligible impressions we show at the absolute top of the ads',
    'Direct parity vs Expedia: US origins, win+tie rate',
    'US outbound share: ticket share for US origins, international destinations',
    'Merchant share: ticket share where Priceline is merchant of record',
    'Deal share: share of tickets sold as a deal',
    'Conversion: GA4, converted customers / engaged customers',
    'Bookability: acceptance rate, includes multiple attempts by same user',
    'Upsell: share of tickets upsold when there was an upsell opportunity',
    'Booking rate: net tickets/gross tickets',
    'Tickets/order: net tickets/net orders'
]
for note in others_notes:
    word_document.add_paragraph(note, style='List Bullet')
    set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(7), bold=False)
word_document.add_paragraph('\n')

word_document.add_paragraph(f'ROI')
set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(18), bold=True)

# Add ROI Section
# roi_notes = [
#     'ROI = Contribution / Cost',
#     'SEM Brand includes all products, weighted by ratio of flight orders',
#     'Meta includes Kayak/Momondo, includes Kayak credit',
#     'Last week'
# ]
# create_roi_table(word_document, df_roi_v)

for note in notes:
        word_document.add_paragraph(note, style='List Bullet')
        set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(7), bold=False)
word_document.add_paragraph('\n')


# Add Revenue Tables for Business, Carrier, Channel, and Source
tables = [
    (df_business_rev, "Business", business_logos, [
        'All Priceline tickets (includes Express Deals and Phone Sales, not normalized)'
    ]),
    (df_carrier_rev, "Carrier", carrier_logos, [
        'Priceline (including phone sales), B2C, Standalone (not normalized)'
    ]),
    (df_channel_rev, "Channel", channel_logos, [
        'Priceline (including phone sales), B2C, Standalone (includes Express Deals, not normalized)'
    ]),
    (df_source_rev, "Source", source_logos, [
        'Priceline (including phone sales), B2C, Standalone (includes Express Deals, not normalized)',
        'Indirect Connect = GDS + Aggregators + Consolidators + NDC-X'
    ])
]
for df, title, logos, notes in tables:
    if df is None:
        continue

   # Add specific custom notes for each table
    if title == "Business":
        custom_note = "\nTotal Business - Net Contr+fee"
    elif title == "Carrier":
        custom_note = "\nPriceline B2C Standalone - Net Contr+fee"
    else:
        custom_note = ""

    # Add the custom note before the table
    word_document.add_paragraph(custom_note)
    set_font(word_document.paragraphs[-1], font_name="Montserrat", font_size=Pt(18), bold=True)
    word_document.add_paragraph()

    create_word_table(df, title, logos, word_document)


# Save the Document
output_filename = os.path.join('../output/', f'Flight Performance Week Ending {end_date}.docx')
word_document.save(output_filename)
print(f"Word document '{output_filename}' has been created successfully.")

# Save PDF to Shared Drive
share_drive_path = '../../../Flight Weekly Note Output/'
word_document.save(share_drive_path + f'Flight Performance Week Ending {end_date}.pdf')

print(f"Word document saved to shared drive at '{share_drive_path}' has been created successfully.")
